In [20]:
!rm -rf /content/KernelOracle

In [21]:
%cd /content

!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/mmarcotullio/KernelOracle.git
%cd /content/KernelOracle
!git checkout mansheel-updates
!git status
!ls

/content
Cloning into 'KernelOracle'...
remote: Enumerating objects: 1036, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 1036 (delta 9), reused 25 (delta 4), pack-reused 1001 (from 1)
Receiving objects: 100% (1036/1036), 79.45 MiB | 27.39 MiB/s, done.
Resolving deltas: 100% (258/258), done.
Encountered 1 file(s) that should have been pointers, but weren't:
	TCN/data/collect_traces_docs.txt
/content/KernelOracle
error: Your local changes to the following files would be overwritten by checkout:
	TCN/data/collect_traces_docs.txt
Please commit your changes or stash them before you switch branches.
Aborting
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   TCN/data/collect_traces_docs.txt

no changes added to commit (use "git add" and/or

In [22]:
!git restore TCN/data/collect_traces_docs.txt
!git checkout mansheel-updates
!git branch

Encountered 1 file(s) that should have been pointers, but weren't:
	TCN/data/collect_traces_docs.txt
error: Your local changes to the following files would be overwritten by checkout:
	TCN/data/collect_traces_docs.txt
Please commit your changes or stash them before you switch branches.
Aborting
* main


In [23]:
!git reset --hard
!git checkout mansheel-updates
!git branch

HEAD is now at 1328a97 Merge branch 'main' of github.com:mmarcotullio/KernelOracle
Encountered 1 file(s) that should have been pointers, but weren't:
	TCN/data/collect_traces_docs.txt
error: Your local changes to the following files would be overwritten by checkout:
	TCN/data/collect_traces_docs.txt
Please commit your changes or stash them before you switch branches.
Aborting
* main


In [24]:
!rm -f /content/KernelOracle/TCN/data/collect_traces_docs.txt
!git checkout -f mansheel-updates
!git branch

Branch 'mansheel-updates' set up to track remote branch 'mansheel-updates' from 'origin'.
Switched to a new branch 'mansheel-updates'
  main
* mansheel-updates


In [25]:
!pip install -q torch pandas numpy scikit-learn matplotlib tqdm

In [26]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import os

DATA_DIR = "/content/drive/MyDrive/data_tcn"
print(os.listdir(DATA_DIR))

['all.csv', 'test_seen.csv', 'test_unseen.csv', 'train.csv']


In [28]:
!ln -s /content/drive/MyDrive/data_tcn /content/KernelOracle/data_tcn
!ls -lh /content/KernelOracle/data_tcn

lrwxrwxrwx 1 root root 31 Mar  8 01:21 /content/KernelOracle/data_tcn -> /content/drive/MyDrive/data_tcn


In [29]:
%%bash
cd /content/KernelOracle

python - <<'PY'
from pathlib import Path

path = Path("/content/KernelOracle/tcn/scripts/preprocess.py")
text = path.read_text()

old = '''def encode_df(df: pd.DataFrame, vocab: Dict) -> pd.DataFrame:
    df = df.copy()

    df["pid_str"] = df["pid"].astype(str)
    df["pid_idx"] = df["pid_str"].map(vocab["pid_to_idx"]).astype(np.int64)

    df["prev_state_str"] = df["prev_state"].astype(str).fillna("UNK")

    if "UNK" not in vocab["state_to_idx"]:

        vocab["state_to_idx"]["UNK"] = len(vocab["state_to_idx"])
        vocab["idx_to_state"][vocab["state_to_idx"]["UNK"]] = "UNK"

    df["state_idx"] = df["prev_state_str"].map(lambda s: vocab["state_to_idx"].get(s, vocab["state_to_idx"]["UNK"])).astype(np.int64)


    df["cpu"] = pd.to_numeric(df.get("cpu", 0), errors="coerce").fillna(0.0).astype(np.float32)
    df["time_diff"] = pd.to_numeric(df.get("time_diff", 0), errors="coerce").fillna(0.0).astype(np.float32)


    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    return df
'''

new = '''def encode_df(df: pd.DataFrame, vocab: Dict) -> pd.DataFrame:
    df = df.copy()

    df["pid_str"] = df["pid"].astype(str)

    if "UNK" not in vocab["pid_to_idx"]:
        vocab["pid_to_idx"]["UNK"] = len(vocab["pid_to_idx"])
        vocab["idx_to_pid"][vocab["pid_to_idx"]["UNK"]] = "UNK"

    df["pid_idx"] = df["pid_str"].map(
        lambda p: vocab["pid_to_idx"].get(p, vocab["pid_to_idx"]["UNK"])
    ).astype(np.int64)

    df["prev_state_str"] = df["prev_state"].astype(str).fillna("UNK")

    if "UNK" not in vocab["state_to_idx"]:
        vocab["state_to_idx"]["UNK"] = len(vocab["state_to_idx"])
        vocab["idx_to_state"][vocab["state_to_idx"]["UNK"]] = "UNK"

    df["state_idx"] = df["prev_state_str"].map(
        lambda s: vocab["state_to_idx"].get(s, vocab["state_to_idx"]["UNK"])
    ).astype(np.int64)

    df["cpu"] = pd.to_numeric(df.get("cpu", 0), errors="coerce").fillna(0.0).astype(np.float32)
    df["time_diff"] = pd.to_numeric(df.get("time_diff", 0), errors="coerce").fillna(0.0).astype(np.float32)

    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    return df
'''

if old not in text:
    raise SystemExit("Could not find the expected encode_df block. Stop here and paste the file contents.")
path.write_text(text.replace(old, new))
print("Patched preprocess.py successfully.")
PY

rm -rf /content/KernelOracle/tcn/artifacts
mkdir -p /content/KernelOracle/tcn/artifacts

python /content/KernelOracle/tcn/scripts/preprocess.py \
  --csv /content/KernelOracle/data_tcn/train.csv \
  --out /content/KernelOracle/tcn/artifacts/train.npz \
  --vocab_out /content/KernelOracle/tcn/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4

python /content/KernelOracle/tcn/scripts/preprocess.py \
  --csv /content/KernelOracle/data_tcn/test_seen.csv \
  --out /content/KernelOracle/tcn/artifacts/test_seen.npz \
  --vocab_out /content/KernelOracle/tcn/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

python /content/KernelOracle/tcn/scripts/preprocess.py \
  --csv /content/KernelOracle/data_tcn/test_unseen.csv \
  --out /content/KernelOracle/tcn/artifacts/test_unseen.npz \
  --vocab_out /content/KernelOracle/tcn/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

ls -lh /content/KernelOracle/tcn/artifacts

Patched preprocess.py successfully.
Saved /content/KernelOracle/tcn/artifacts/train.npz
Windows: 1812164, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/tcn/artifacts/vocab.json
Saved /content/KernelOracle/tcn/artifacts/test_seen.npz
Windows: 792463, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/tcn/artifacts/vocab.json
Saved /content/KernelOracle/tcn/artifacts/test_unseen.npz
Windows: 2172560, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/tcn/artifacts/vocab.json
total 121M
-rw-r--r-- 1 root root  16M Mar  8 01:23 test_seen.npz
-rw-r--r-- 1 root root  55M Mar  8 01:26 test_unseen.npz
-rw-r--r-- 1 root root  51M Mar  8 01:23 train.npz
-rw-r--r-- 1 root root 124K Mar  8 01:22 vocab.json


In [30]:
%%writefile /content/KernelOracle/tcn/models/lstm.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn


@dataclass
class LSTMConfig:
    num_pids: int
    num_states: int
    pid_emb: int = 32
    state_emb: int = 8
    cont_dim: int = 2
    hidden: int = 128
    num_layers: int = 2
    dropout: float = 0.1


class LSTMNextPid(nn.Module):
    def __init__(self, cfg: LSTMConfig):
        super().__init__()
        self.cfg = cfg

        self.pid_emb = nn.Embedding(cfg.num_pids, cfg.pid_emb)
        self.state_emb = nn.Embedding(cfg.num_states, cfg.state_emb)

        in_feat = cfg.pid_emb + cfg.state_emb + cfg.cont_dim

        self.lstm = nn.LSTM(
            input_size=in_feat,
            hidden_size=cfg.hidden,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=(cfg.dropout if cfg.num_layers > 1 else 0.0),
        )

        self.head = nn.Linear(cfg.hidden, cfg.num_pids)

    def forward(
        self,
        pid: torch.Tensor,
        cont: torch.Tensor,
        state: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        if state is None:
            raise ValueError("state tensor is required")

        pid_e = self.pid_emb(pid)      # (B, L, pid_emb)
        st_e = self.state_emb(state)   # (B, L, state_emb)

        x = torch.cat([pid_e, st_e, cont], dim=-1)   # (B, L, in_feat)
        out, _ = self.lstm(x)                        # (B, L, hidden)

        last = out[:, -1, :]                         # (B, hidden)
        logits = self.head(last)                     # (B, num_pids)
        return logits

Writing /content/KernelOracle/tcn/models/lstm.py


In [31]:
%%writefile /content/KernelOracle/tcn/train_lstm.py
from __future__ import annotations

import argparse
import os
import time
from typing import Dict

import torch
from torch.utils.data import DataLoader

from tcn.models.lstm import LSTMConfig, LSTMNextPid
from tcn.utils.data import TraceWindowDataset, Vocab, batch_to_device
from tcn.utils.metrics import top1_accuracy


def make_loader(npz_path: str, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TraceWindowDataset(npz_path)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )


def train_one_epoch(model: torch.nn.Module,
                    loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    n = 0

    for batch in loader:
        batch = batch_to_device(batch, device)
        logits = model(batch["pid"], batch["cont"], state=batch["state"])
        loss = torch.nn.functional.cross_entropy(logits, batch["y"])

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * batch["y"].shape[0]
        n += batch["y"].shape[0]

    return total_loss / max(1, n)


@torch.no_grad()
def eval_model(model: torch.nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    acc_sum = 0.0
    n = 0

    for batch in loader:
        batch = batch_to_device(batch, device)
        logits = model(batch["pid"], batch["cont"], state=batch["state"])
        acc = top1_accuracy(logits, batch["y"])
        bsz = batch["y"].shape[0]
        acc_sum += acc * bsz
        n += bsz

    return {"acc": acc_sum / max(1, n)}


@torch.no_grad()
def measure_inference_latency_ms_safe(model: torch.nn.Module,
                                      batch: Dict[str, torch.Tensor],
                                      warmup: int = 20,
                                      iters: int = 100) -> float:
    model.eval()

    def _sync():
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    for _ in range(warmup):
        _ = model(batch["pid"], batch["cont"], state=batch["state"])

    _sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = model(batch["pid"], batch["cont"], state=batch["state"])
    _sync()
    t1 = time.perf_counter()

    return (t1 - t0) * 1000.0 / iters


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_npz", default="tcn/artifacts/train.npz")
    ap.add_argument("--test_seen_npz", default="tcn/artifacts/test_seen.npz")
    ap.add_argument("--test_unseen_npz", default="tcn/artifacts/test_unseen.npz")
    ap.add_argument("--vocab", default="tcn/artifacts/vocab.json")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--hidden", type=int, default=128)
    ap.add_argument("--num_layers", type=int, default=2)
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--save_dir", default="tcn/models")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.save_dir, exist_ok=True)

    vocab = Vocab.load(args.vocab)
    cfg = LSTMConfig(
        num_pids=len(vocab.pid_to_idx),
        num_states=len(vocab.state_to_idx),
        hidden=args.hidden,
        num_layers=args.num_layers,
        dropout=args.dropout,
    )

    model = LSTMNextPid(cfg).to(device)

    train_loader = make_loader(args.train_npz, args.batch_size, shuffle=True)
    seen_loader = make_loader(args.test_seen_npz, args.batch_size, shuffle=False) if os.path.exists(args.test_seen_npz) else None
    unseen_loader = make_loader(args.test_unseen_npz, args.batch_size, shuffle=False) if os.path.exists(args.test_unseen_npz) else None

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)

    best_seen = -1.0
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        loss = train_one_epoch(model, train_loader, optimizer, device)
        dt = time.time() - t0

        msg = f"epoch={epoch} loss={loss:.4f} time={dt:.1f}s"

        if seen_loader is not None:
            seen = eval_model(model, seen_loader, device)["acc"]
            msg += f" | seen_acc={seen:.4f}"
        else:
            seen = None

        if unseen_loader is not None:
            unseen = eval_model(model, unseen_loader, device)["acc"]
            msg += f" | unseen_acc={unseen:.4f}"

        print(msg)

        if seen is not None and seen > best_seen:
            best_seen = seen
            ckpt_path = os.path.join(args.save_dir, "lstm_nextpid_best.pt")
            torch.save({"cfg": cfg.__dict__, "state_dict": model.state_dict()}, ckpt_path)
            print(f"Saved best checkpoint -> {ckpt_path}")

    batch = next(iter(train_loader))
    batch = batch_to_device(batch, device)
    lat_ms = measure_inference_latency_ms_safe(
        model,
        {"pid": batch["pid"], "cont": batch["cont"], "state": batch["state"]},
        warmup=20,
        iters=100
    )
    print(f"Avg forward latency per batch: {lat_ms:.3f} ms (device={device})")


if __name__ == "__main__":
    main()

Writing /content/KernelOracle/tcn/train_lstm.py


In [32]:
%%bash
cd /content/KernelOracle

python - <<'PY'
import json
from pathlib import Path

vocab_path = Path("/content/KernelOracle/tcn/artifacts/vocab.json")
with open(vocab_path, "r") as f:
    vocab = json.load(f)

changed = False

if "UNK" not in vocab["pid_to_idx"]:
    unk_idx = len(vocab["pid_to_idx"])
    vocab["pid_to_idx"]["UNK"] = unk_idx
    vocab["idx_to_pid"][str(unk_idx)] = "UNK"
    changed = True
    print(f"Added PID UNK -> {unk_idx}")

if "UNK" not in vocab["state_to_idx"]:
    unk_idx = len(vocab["state_to_idx"])
    vocab["state_to_idx"]["UNK"] = unk_idx
    vocab["idx_to_state"][str(unk_idx)] = "UNK"
    changed = True
    print(f"Added STATE UNK -> {unk_idx}")

if changed:
    with open(vocab_path, "w") as f:
        json.dump(vocab, f)
    print("Updated vocab.json")
else:
    print("vocab.json already has UNK entries")
PY

rm -f /content/KernelOracle/tcn/artifacts/train.npz
rm -f /content/KernelOracle/tcn/artifacts/test_seen.npz
rm -f /content/KernelOracle/tcn/artifacts/test_unseen.npz

python /content/KernelOracle/tcn/scripts/preprocess.py \
  --csv /content/KernelOracle/data_tcn/train.csv \
  --out /content/KernelOracle/tcn/artifacts/train.npz \
  --vocab_out /content/KernelOracle/tcn/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

python /content/KernelOracle/tcn/scripts/preprocess.py \
  --csv /content/KernelOracle/data_tcn/test_seen.csv \
  --out /content/KernelOracle/tcn/artifacts/test_seen.npz \
  --vocab_out /content/KernelOracle/tcn/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

python /content/KernelOracle/tcn/scripts/preprocess.py \
  --csv /content/KernelOracle/data_tcn/test_unseen.csv \
  --out /content/KernelOracle/tcn/artifacts/test_unseen.npz \
  --vocab_out /content/KernelOracle/tcn/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

ls -lh /content/KernelOracle/tcn/artifacts

Added PID UNK -> 3802
Added STATE UNK -> 7
Updated vocab.json
Saved /content/KernelOracle/tcn/artifacts/train.npz
Windows: 1812164, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/tcn/artifacts/vocab.json
Saved /content/KernelOracle/tcn/artifacts/test_seen.npz
Windows: 792463, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/tcn/artifacts/vocab.json
Saved /content/KernelOracle/tcn/artifacts/test_unseen.npz
Windows: 2172560, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/tcn/artifacts/vocab.json
total 121M
-rw-r--r-- 1 root root  16M Mar  8 01:29 test_seen.npz
-rw-r--r-- 1 root root  55M Mar  8 01:31 test_unseen.npz
-rw-r--r-- 1 root root  51M Mar  8 01:28 train.npz
-rw-r--r-- 1 root root 124K Mar  8 01:27 vocab.json


In [33]:
!python -m tcn.train_lstm \
  --train_npz /content/KernelOracle/tcn/artifacts/train.npz \
  --test_seen_npz /content/KernelOracle/tcn/artifacts/test_seen.npz \
  --test_unseen_npz /content/KernelOracle/tcn/artifacts/test_unseen.npz \
  --vocab /content/KernelOracle/tcn/artifacts/vocab.json \
  --epochs 1 \
  --batch_size 128 \
  --lr 0.001 \
  --hidden 128 \
  --num_layers 2 \
  --dropout 0.1 \
  --save_dir /content/KernelOracle/tcn/models

epoch=1 loss=4.8373 time=6182.1s | seen_acc=0.0426 | unseen_acc=0.4076
Saved best checkpoint -> /content/KernelOracle/tcn/models/lstm_nextpid_best.pt
Avg forward latency per batch: 128.825 ms (device=cpu)
